In [2]:
import kagglehub
import altair as alt
import pandas as pd
# Download latest version
path = kagglehub.dataset_download("rush4ratio/video-game-sales-with-ratings")

print("Path to dataset files:", path)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/cjc/.cache/kagglehub/datasets/rush4ratio/video-game-sales-with-ratings/versions/2


In [3]:
import pandas as pd
#| label: 0-Dataset 
vg_sales = pd.read_csv(f"{path}/Video_Games_Sales_as_at_22_Dec_2016.csv")
categorical_cols = ['Name', 'Platform', 'Genre', 'Publisher', 'Developer', 'Rating']
numeric_cols = ['Year_of_Release', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales', 'Critic_Score', 'Critic_Count', 'User_Score', 'User_Count']
# Convert categorical to string, and numeric to numeric, where bad values are made NaN.
vg_sales[categorical_cols] = vg_sales[categorical_cols].astype('string')
vg_sales[numeric_cols] = vg_sales[numeric_cols].apply(pd.to_numeric, errors='coerce')
# Remove all rows that have at least one NaN value in them.
vgs_cleaned = vg_sales.dropna()
vgs_cleaned = vg_sales.dropna(subset=['User_Score', 'Critic_Score', 'Rating', 'Publisher'])
vgs_cleaned = vgs_cleaned[~vgs_cleaned['Rating'].isin(['RP', 'K-A', 'AO'])]

vgs_cleaned


,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8.0,322.0,Nintendo,E
2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E
3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8.0,192.0,Nintendo,E
6,New Super Mario Bros.,DS,2006.0,Platform,Nintendo,11.28,9.14,6.50,2.88,29.80,89.0,65.0,8.5,431.0,Nintendo,E
7,Wii Play,Wii,2006.0,Misc,Nintendo,13.96,9.18,2.93,2.84,28.92,58.0,41.0,6.6,129.0,Nintendo,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16667,E.T. The Extra-Terrestrial,GBA,2001.0,Action,NewKidCo,0.01,0.00,0.00,0.00,0.01,46.0,4.0,2.4,21.0,Fluid Studios,E
16677,Mortal Kombat: Deadly Alliance,GBA,2002.0,Fighting,Midway Games,0.01,0.00,0.00,0.00,0.01,81.0,12.0,8.8,9.0,Criterion Games,M
16696,Metal Gear Solid V: Ground Zeroes,PC,2014.0,Action,Konami Digital Entertainment,0.00,0.01,0.00,0.00,0.01,80.0,20.0,7.6,412.0,Kojima Productions,M
16700,Breach,PC,2011.0,Shooter,Destineer,0.01,0.00,0.00,0.00,0.01,61.0,12.0,5.8,43.0,Atomic Games,T


In [4]:
def group_publisher(pub):
    pub_lower = pub.lower()
    if pub_lower.startswith('nintendo'):
        return 'Nintendo'
    elif pub_lower.startswith('namco'):
        return 'Namco'
    elif pub_lower.startswith('ubisoft'):
        return 'Ubisoft'
    elif pub_lower.startswith('activision'):
        return 'Activision'
    elif pub_lower.startswith('electronic arts'):
        return 'Electronic Arts'
    elif pub_lower.startswith('sega'):
        return 'Sega'
    elif pub_lower.startswith('digital entertainment'):
        return 'Digital Entertainment'
    elif pub_lower.startswith('rockstar'):
        return 'Rockstar'
    elif pub_lower.startswith('sony'):
        return 'Sony'
    else:
        return 'Other'

vgs_cleaned = vgs_cleaned.copy()
vgs_cleaned['Publisher_Group'] = vgs_cleaned['Publisher'].apply(group_publisher)
vgs_cleaned_mosaic = vgs_cleaned.groupby(['Publisher_Group', 'Genre']).agg(
    count=('Name', 'count'),
    platform=('Platform', lambda x: x.value_counts().index[0])
).reset_index()
vgs_cleaned_mosaic.columns = ['publisher', 'genre', 'count', 'platform']
vgs_cleaned_mosaic.to_json('vgs_cleaned.json', orient='records')

In [5]:
#| label: 1-Critic vs User Score Scatter Plot with Multiple Filters
import pandas as pd
import altair as alt

## dropdown selections
developers = sorted(vgs_cleaned['Developer'].dropna().unique().tolist())
platforms  = sorted(vgs_cleaned['Platform'].dropna().unique().tolist())

developer_options = ['All'] + developers
platform_options = ['All'] + platforms

## dropdown params
developer_select = alt.param(
    name='Developer',
    bind=alt.binding_select(options=developer_options, name='Developer: '),
    value='All'
)

platform_select = alt.param(
    name='Platform',
    bind=alt.binding_select(options=platform_options, name='Platform: '),
    value='All'
)

## filter conditions
developer_filter = (developer_select == 'All') | (alt.datum.Developer == developer_select)
platform_filter = (platform_select == 'All')   | (alt.datum.Platform == platform_select)

## legend selections
rating_genre_select = alt.selection_point(
    fields=['Rating', 'Genre'],
    empty='all' 
)

## slider selection
sales_slider = alt.param(
    name='Global_Sales',
    value=0.0,
    bind=alt.binding_range(
        min=0,
        max=35,
        step=0.05,
        name='Min Global Sales in Millions: '
    )
)

# slider filter
sales_filter = alt.datum.Global_Sales >= sales_slider

# common filter to reuse on both charts
common_filter = (
    developer_filter &
    platform_filter &
    sales_filter
)

alt.data_transformers.enable('default', max_rows=100000)

## build the scatter plot
scatter_plot = (alt.Chart(vgs_cleaned).transform_calculate(
    User_Score_100='datum.User_Score * 10'
    ).mark_circle(
        size=40
    ).encode(
    x = alt.X('Critic_Score:Q', title='Critic Score (%)'),
    y = alt.Y('User_Score_100:Q', title='User Score (%)'),
    color = alt.Color('Genre:N', title='Content Rating', legend=None),
    size = alt.Size('Global_Sales:Q',
                    scale=alt.Scale(range=[10, 500]),
                    title='Global Sales (M)',
                    legend=alt.Legend(
                      orient='right',
                      offset=-50)
                    ),
    opacity = alt.condition(
        rating_genre_select,
        alt.value(1),
        alt.value(0)
        ),
        tooltip=[
            'Name',
            'Year_of_Release',
            'Genre',
            'Platform',
            'Rating',
            'Developer',
            'Global_Sales',
            'Critic_Count',
            'User_Count'
        ]
    ).properties(
        width=800,
        height=400,
        title='User vs Critic Score by Platform, Developer, Genre, and Rating'
    ).transform_filter(
        common_filter
    ).transform_filter(
        rating_genre_select
    )
)
    
## build the bar chart
bar_chart = (alt.Chart(vgs_cleaned).transform_fold(
        ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales'],
        as_ =['Region', 'Sales']
    ).mark_bar().encode(
        x = alt.X('Region:N', title='Region', sort= 'y', axis=alt.Axis(labelAngle=0)),
        y = alt.Y('sum(Sales):Q', title='Total Sales in Millions'),
        color = alt.Color('Region:N', title='Region', legend=None),
        tooltip = [
            'Region:N',
            alt.Tooltip('sum(Sales):Q', title='Total Sales in Millions')
        ]
    ).add_params(
        developer_select,
        platform_select,
        sales_slider
    ).transform_filter(
        common_filter
    ).transform_filter(
        rating_genre_select
    ).properties(
        width=500,
        height=250,
        title='Regional Sales of Games by Platform, Developer, Genre, and Rating'
    )
)
    
    ## build the legend chart
legend = (
    alt.Chart(vgs_cleaned)
    .mark_rect()
    .encode(
        y = alt.Y('Genre:N', title='Genre'),
        x = alt.X(
            'Rating:N', 
            title='Content Rating', 
            sort=['E', 'E10+', 'T', 'M'], 
            axis=alt.Axis(labelAngle=0)
            ),
        color = alt.condition(
            rating_genre_select,
            alt.Color('Genre:N', legend=None),
            alt.value('lightgray')
        ),
        opacity = alt.condition(
            rating_genre_select,
            alt.value(1),
            alt.value(0.1)
    ))
    .add_params(rating_genre_select)
    .properties(
        width=250,
        height=250,
        title='Filter by Rating & Genre'
    )
)

## Formatting ## 
# Put scatter on top, bar chart and legend on bottom
bottom_row = alt.hconcat(
    bar_chart,
    legend
).resolve_scale(
    color='independent'
)

final_chart = (
    alt.vconcat(
        scatter_plot,
        bottom_row
    )
    .configure_concat(spacing=20)
    .configure_view(stroke=None)
    .configure_axis(labelFontSize=11, titleFontSize=13)
    .configure_title(fontSize=16, anchor='start')
)

final_chart
final_chart.save('scatterplot_with_barchart.html')

#| label: 1-Context for Scatter Plot with Multiple Filters 

This visualization displays the relationship between Critic Score and User Score for each video game in the dataset using a scatterplot. Each point mark represents a single game. The position channels (x = critic score, y = user score) encode two quantitative attributes (on a 0-100 scale) to help users identify correlations, clusters, and outliers across the industry. The color channel encodes the game’s ESRB (Entertainment Software Rating Board) content rating (a categorical attribute), while the size channel represents global sales in millions to emphasize games with larger sale impact.

To support exploration, the visualization provides multiple interactive filters, including genre, rating, developer, platform, and minimum global sales. This allows users to search for specific subsets of the data or to compare groups. This makes the tool useful for high-level analysis tasks such as discovering which types of games tend to receive high ratings from both critics and players, identifying market trends, or examining how content rating relates to commercial performance.

Notably, games rated E (Everyone) show the highest sales overall, likely due to their broader accessibility.

In [21]:
#| label: 2-Revenue Bar Chart

# get list of genres for dropdowns
genres = sorted(vgs_cleaned['Genre'].dropna().unique().tolist())

# Dropdown params for two genres
genre1_select = alt.param(
    name='Genre1',
    bind=alt.binding_select(options=genres, name='Genre 1: '),
    value=genres[0]
)

genre2_select = alt.param(
    name='Genre2',
    bind=alt.binding_select(options=genres, name='Genre 2: '),
    value=genres[1] if len(genres) > 1 else genres[0]
)

# keep only rows from the two selected genres
two_genre_filter = (
    (alt.datum.Genre == genre1_select) |
    (alt.datum.Genre == genre2_select)
)

# Boxplot comparing user score between the two genres
user_score_box = (
    alt.Chart(vgs_cleaned).transform_calculate(
    User_Score_100='datum.User_Score * 10'
    ).mark_boxplot(size=60)
    .encode(
        x = alt.X('Genre:N', title='Genre', axis=alt.Axis(labelAngle=0)),
        y = alt.Y('User_Score_100:Q', title='User Review Score (%)'),
        color = alt.Color('Genre:N', title='Genre', legend=None)
    )
    .transform_filter(two_genre_filter)
    .add_params(genre1_select, genre2_select)
    .properties(
        width=325,
        height=300,
        title='User Score Distribution for Two Genres'
    )
)

# Boxplot comparing critic scores between the two genres
critic_score_box = (
    alt.Chart(vgs_cleaned).mark_boxplot(size=60)
    .encode(
        x = alt.X('Genre:N', title='Genre', axis=alt.Axis(labelAngle=0), ),
        y = alt.Y('Critic_Score:Q', title='Critic Review Score (%)'),
        color = alt.Color('Genre:N', title='Genre', legend=None)
    )
    .transform_filter(two_genre_filter)
    .add_params(genre1_select, genre2_select)
    .properties(
        width=300,
        height=300,
        title='Critic Score Distribution for Two Genres'
    )
)

top_row = critic_score_box | user_score_box

# Summary table of average and total user and critic scores for the two genres
summary = (
    alt.Chart(vgs_cleaned)
    .transform_calculate(
        User_Score_100='datum.User_Score * 10'
    )
    .transform_filter(two_genre_filter)
    .transform_aggregate(
        avg_user_score='mean(User_Score_100)',
        total_user_count='sum(User_Count)',
        avg_critic_score='mean(Critic_Score)',
        total_critic_count='sum(Critic_Count)',
        avg_global_sales='mean(Global_Sales)',
        groupby=['Genre']
    )
    # fold the internal metric names
    .transform_fold(
        [
            'avg_user_score',
            'total_user_count',
            'avg_critic_score',
            'total_critic_count',
            'avg_global_sales'
        ],
        as_=['Metric', 'Value']
    )
    # map internal names for labels
    .transform_calculate(
        Labels=(
            "datum.Metric == 'avg_user_score' ? 'Average User Score' : "
            "datum.Metric == 'total_user_count' ? 'Total User Reviews' : "
            "datum.Metric == 'avg_critic_score' ? 'Average Critic Score' : "
            "datum.Metric == 'total_critic_count' ? 'Total Critic Reviews' : "
            "'Average Global Sales in Millions'"
        )
    )
    .mark_text(fontSize=11)
    .encode(
        x = alt.X('Genre:N', title='Genre', axis=alt.Axis(labelAngle=0)),
        y = alt.Y(
            'Labels:N',
            title=None,
            sort=[
                'Average User Score',
                'Total User Reviews',
                'Average Critic Score',
                'Total Critic Reviews',
                'Average Global Sales in Millions'
            ],
        ),
        text = alt.Text('Value:Q', format=',.3f'),
        color = alt.Color('Genre:N', legend=None)
    )
    .properties(
        width=300,
        height=110,
        title='Summary table: Average & Total User & Critic Counts'
    )
)

spacer = alt.Chart().mark_text().encode().properties(width=63)

centered_summary = alt.hconcat(
    spacer,
    summary,
).resolve_legend(color="independent")

final_chart = alt.vconcat(
    top_row,
    centered_summary
).configure_view(
    stroke=None
)

final_chart.save('boxplot_with_summary.html')

#| label: 2-Context for Revenue Bar Chart

This visualization summarizes the total video game sales across five regions: Japan, Europe, North America, Other regions, and Global totals. Each bar mark represents a region, using length/height to show total sales volume. Color is used as a categorical channel to distinguish the regions from one another.

To support exploration, the visualization includes interactive filters for genre, ESRB (Entertainment Software Rating Board) rating, developer, and platform, allowing users to refine the view and compare how regional sales distributions change across different subsets of games. This makes the chart useful for identifying market patterns and understanding how particular categories of games perform in different geographic areas.

Overall and among individual regions, North America generates the highest revenue, followed by Europe, while Japan consistently shows the lowest total sales within this dataset.

In [7]:
#| label: 3-Global Sales over time.
alt.data_transformers.enable('default', max_rows=100000)

click = alt.selection_point(fields=['Genre'])
hover = alt.selection_point(on='mouseover', fields=['Genre'], empty=False)

interactive_time_plot = (
    alt.Chart(vgs_cleaned)
    .mark_line()
    .encode(
        x=alt.X('Year_of_Release:O', title='Year'),
        y=alt.Y('sum(Global_Sales):Q', title='Total Global Sales (Millions)'),
        color=alt.Color('Genre:N'),
        opacity=alt.condition(click, alt.value(1), alt.value(0.15))
    )
    .add_params(click)
)
points = (
    alt.Chart(vgs_cleaned)
    .mark_circle()
    .encode(
        x='Year_of_Release:O',
        y='sum(Global_Sales):Q',
        color='Genre:N',
        opacity=alt.condition(click, alt.value(1), alt.value(0.15)),
        size=alt.condition(
            hover,
            alt.value(200),  
            alt.value(40)     
        ),
        tooltip=[
            'Year_of_Release',
            'Genre',
            alt.Tooltip('sum(Global_Sales):Q', title='Total Global Sales')
        ]
    )
    .add_params(click, hover)
)
timeline_plot = (interactive_time_plot + points).properties(
    width=900,
    height=500,
    title='Global Sales Over Time by Genre'
)

timeline_plot
timeline_plot.save('linechart.html')


#| label: 3-Global Sales over time description.

This visualization in a line chart for the year and the total global sales in millions of dollars for that year. Each line on the line chard represents a genre of a video game. 

From the line chart we can see that over the years, all genres have fallen in popularity while shooter, sports, and action games have risen on-net. The only exception is the sudden plunder in 2016, but this can be explained by the fact that the data was scraped from a blog, so perhaps the blog website became less popular and outdated.

In [8]:
#| label: 4-Global Sales for genre and platform.
import altair as alt
alt.data_transformers.enable('default', max_rows=100000)

genre_platform_plot = (
    alt.Chart(vgs_cleaned)
    .mark_circle(opacity=0.7)
    .encode(
        x=alt.X('Genre:O', title='Genre'),
        y=alt.Y('Platform:N', title='Platform'),
        size=alt.Size(
            'Global_Sales:Q',
            scale=alt.Scale(range=[20, 600]),
            title='Global Sales (Millions)'
        ),
        color=alt.Color('Genre:N', title='Genre'),
        tooltip=[
            'Name',
            'Genre',
            'Platform',
            'Year_of_Release',
            'Global_Sales',
            'Critic_Score',
            'User_Score'
        ]
    )
    .properties(
        width=900,
        height=600,
        title='Genre vs Platform (Bubble Size = Global Sales)'
    )
)

genre_platform_plot
genre_platform_plot.save('bubblechart.html')

#| label: 4-Global Sales for genre and platform description.

The above visualization is a bubble chart between video game genre and platform, where the bubble size corresponds to global sales in millions of dollars.

What we can see from the visualization is that different platforms have seen different levels of success across genres, suggesting that perhaps those specific platofrms are godo for the specific genre of game (e.g. Wii for sports games), or the games the platform made were good for the genre. Perhaps a combination of both.

### HTML File Generator

Uses qarto plugin to generate html files from labeled cells for visualizations.
Main steps are:
1. Define a dictionary tying file names to labels. Keys, that is labels, are defined by placing \<#| label: \<chart number\>-\<description\>\>.
2. Create a dictionary of the cells and their cell types (markdown or code).
3. Define the quarto file style for all htmls.
4. Create quarto files and render them into html files.
5. Run the process.


In [9]:
# Import plugins
import nbformat
import subprocess
from pathlib import Path
import shutil

# Can add descriptive name here. For filename, name is concatenatenated, lowercased, and underscored.
file_dict: dict[int, str] = {
    0: "Data",
    # 1: "Scatterplot",
    # 2: "Barchart",
    # 3: "Linechart",
    # 4: "Bubblechart"
}

In [10]:
## Gets the cell labels and their contents for given Jupyter notebook as a dictionary between label number, 
## and a list of a dictionary of the cell type which is one of "markdown" or "code" along with the cells contents.
def get_cell_contents(path: str) -> dict[int, list[dict[str, str]]]:
    notebook_node: nbformat.NotebookNode = nbformat.read(path, as_version = 4)
    labeled: dict[int, dict[str, str]] = {}
    # Iterate through each cell, and add to labeled if first line contains the label.
    for cell in notebook_node.cells:
        content: str = cell.source
        first_line: str = content.split("\n")[0].strip()
        try:
            if first_line.startswith("#| label:"):
                full_label: str = first_line.split(":")[1].strip()
                prefix: int = int(full_label.split('-')[0])
                if prefix not in labeled:
                    labeled[prefix] = []
                labeled[prefix].append({
                    "type": cell.cell_type,
                    "content": content,
                    "full-label": full_label
                })
        except:
            print(f"Couldn't parse cell: {first_line}.")
    return labeled

In [11]:
def generate_quarto_contents(code: str, description: str) -> str:
    content: str = f"""---
title: ""
format:
  html:
    code-fold: true
    code-summary: "View Code"
notebook-links: false
---

## Visualization

{code}

## Description

{description}
"""
    return content

In [12]:
## Given the path to the notebook and a dictionary of labels to names,
## renders all of its properly labeled cells (decsribed in dicionary section)
## as html files through quarto.
def cells_to_html(path: str, label_dict: dict[int, str]) -> None:
    # Add files to this folder, and overwrite existing files if share the same path.
    notebook_name: str = Path(path).name
    output_dir: pathlib.Path = Path("build")
    output_dir.mkdir(exist_ok=True)
    # Copy notebook to build folder so that quarto can find it.
    shutil.copy(path, output_dir / notebook_name)
    # Get the cell contents.
    cell_contents: dict[int, list[dict[str: str]]] = get_cell_contents(path)
    for label, file_name in label_dict.items():
        if label not in cell_contents:
            continue
        # For the current label number, checks if exists within the notebooks cell label numbers.
        code_cells: list[dict[str, str]] = [cell_dict for cell_dict in cell_contents[label] if cell_dict['type'] == 'code']
        markdown_cells: list[dict[str, str]] = [cell_dict for cell_dict in cell_contents[label] if cell_dict['type'] == 'markdown']
        # Join the code for the current label.
        def remove_label_line(content: str) -> str:
            lines = content.split('\n', 1)
            return lines[1] if len(lines) > 1 else content
        code_embeds: str = '\n\n'.join(
            f'{{{{< embed {notebook_name}#{c["full-label"]} echo=true >}}}}'
            for c in code_cells
        )
        markdown_embeds: str = '\n\n'.join(
            f'{{{{< embed {notebook_name}#{c["full-label"]} echo=true >}}}}'
            for c in markdown_cells
        )
        # Write the file.
        qmd_path = output_dir / f"{file_name.lower().replace(' ', '_')}.qmd"
        qmd_path.write_text(generate_quarto_contents(code_embeds, markdown_embeds))
        # Render the quarto file as an html.
        result = subprocess.run(['quarto', 'render', str(qmd_path)], check=True)
        # Move html to root, overwriting if necessary.
        html_in_build = output_dir / f'{file_name.lower().replace(" ", "_")}.html'
        html_in_root = Path(f'{file_name.lower().replace(" ", "_")}.html')
        shutil.move(str(html_in_build), str(html_in_root))
    return

In [13]:
if __name__ == "__main__":
    cells_to_html("VisualizationCode.ipynb", file_dict)